<a href="https://colab.research.google.com/github/madhumitha006/Website-RAG-System/blob/main/website_scrabbing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

urls = [
    "https://sece.ac.in/",
    "https://sece.ac.in/department-computer-science-engineering-2/",
    "https://sece.ac.in/placement-training/"
]

data = []

for url in urls:
    try:
        response = requests.get(url, timeout=10)

        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.title.text.strip() if soup.title else "No Title"

        text = soup.get_text(separator=" ", strip=True)

        data.append({
            "title": title,
            "url": url,
            "content": text
        })

        print("Scraped:", url)

    except Exception as e:
        print("Error:", e)

df = pd.DataFrame(data)

df.to_csv("website_data.csv", index=False)

print("CSV Created Successfully")
print(df.head())
print(df.shape)

Scraped: https://sece.ac.in/
Scraped: https://sece.ac.in/department-computer-science-engineering-2/
Scraped: https://sece.ac.in/placement-training/
CSV Created Successfully
                                               title  \
0       SECE: Best Engineering College in Tamil Nadu   
1  Department - Computer Science Engineering - Sr...   
2  Placement Training Programs - Sri Eshwar Engin...   

                                                 url  \
0                                https://sece.ac.in/   
1  https://sece.ac.in/department-computer-science...   
2             https://sece.ac.in/placement-training/   

                                             content  
0  SECE: Best Engineering College in Tamil Nadu S...  
1  Department - Computer Science Engineering - Sr...  
2  Placement Training Programs - Sri Eshwar Engin...  
(3, 3)


In [4]:
!pip install -q langchain-text-splitters

In [5]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

df = pd.read_csv("website_data.csv")

text = " ".join(df["content"].astype(str))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)

print("Total Chunks:", len(chunks))

print("\nFirst Chunk:\n")
print(chunks[0])

Total Chunks: 289

First Chunk:

SECE: Best Engineering College in Tamil Nadu Skip to content Sri Eshwar Engineering College International Internship 2026 @ SECE Campus tour Admission 2025 TNEA CODE 2739 Study In India Alumni ERP Careers IQAC NBA DCP Audited Statements SDG CELL Quick Links Research Ethics Committee Examination Committee Grievance Redressal Committee (GRC) Library Advisory Committee Internal Complaints Committee (ICC) Counsellor Committee (CC) Anti Ragging Committee (ARC) SC/ST Cell AICTE Feedback Contact X HOME


In [6]:
!pip install -q sentence-transformers

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print("Total Embeddings:", len(embeddings))
print("Embedding Dimension:", len(embeddings[0]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total Embeddings: 289
Embedding Dimension: 384


In [10]:
!pip install -q chromadb

In [11]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="sece_collection"
)

for i, chunk in enumerate(chunks):
    collection.add(
        ids=[str(i)],
        documents=[chunk]
    )

print("Data Stored Successfully")
print("Total Records:", collection.count())

Data Stored Successfully
Total Records: 289


In [12]:
query = "placement training"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print(results["documents"][0][0])

a confluence of ideas and new innovations to the forefront. Uniquely designed spaces, inspires students to work on their projects with a free and happy mind, and extracts their best capability. Placement Training Facility As a top placements college, we pride ourselves in providing exclusive placement training facilities. With more than four 200+ seating capacity facilities, four 60+ seating capacity training rooms, we can accommodate about 1000 students to undergo dedicated hands-on placement


In [13]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.0 MB/s eta 0:00:00


In [18]:
from groq import Groq

client = Groq(
    api_key="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"
)

In [19]:
from groq import Groq

client = Groq(api_key="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")

def ask_question(question):

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    context = "\n".join(results["documents"][0])

    prompt = f"""
    Answer the question only from the given context.

    Context:
    {context}

    Question:
    {question}
    """

    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [22]:
query = "What placement training programs are available?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print(results["documents"][0])

['a confluence of ideas and new innovations to the forefront. Uniquely designed spaces, inspires students to work on their projects with a free and happy mind, and extracts their best capability. Placement Training Facility As a top placements college, we pride ourselves in providing exclusive placement training facilities. With more than four 200+ seating capacity facilities, four 60+ seating capacity training rooms, we can accommodate about 1000 students to undergo dedicated hands-on placement', 'Placement Training Programs Industry Institution Interaction Centre for Higher Education & Foreign Language CAMPUS LIFE Life @ Sri Eshwar Flagship Events Student Leadership Council Fine Arts Sports Clubs NCC Community Outreach Student Welfare Library Amenity Centre Medical Centre Hostel Transport ADMISSIONS ERP IQAC ALUMNI CAREER QUICK LINKS Grievance Redressal Committee Anti Ragging Committee Student Welfare Counsellor Committee Anti Ragging Committee SC/ST Cell Contact STUDY IN INDIA X HOM

In [21]:
client = chromadb.PersistentClient(
    path="./chroma_db"
)

In [23]:
collection = client.get_or_create_collection(
    name="sece_collection"
)

print("Collection Created")

Collection Created


In [26]:
collection = client.get_or_create_collection(
    name="sece_collection"
)

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks
)

print("Total Records:", collection.count())

Total Records: 289


In [27]:
collection = client.get_or_create_collection(
    name="sece_collection"
)

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks
)

print("Total Records:", collection.count())

Total Records: 289


In [28]:
results = collection.query(
    query_texts=["placement training"],
    n_results=3
)

print(results["documents"][0][0])

a confluence of ideas and new innovations to the forefront. Uniquely designed spaces, inspires students to work on their projects with a free and happy mind, and extracts their best capability. Placement Training Facility As a top placements college, we pride ourselves in providing exclusive placement training facilities. With more than four 200+ seating capacity facilities, four 60+ seating capacity training rooms, we can accommodate about 1000 students to undergo dedicated hands-on placement


In [29]:
from groq import Groq

client = Groq(
    api_key="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"
)

In [32]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Hello"
        }
    ]
)

print(response.choices[0].message.content)

Hello. How can I assist you today?


In [33]:
response = client.chat.completions.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=[
        {
            "role": "user",
            "content": "Hello"
        }
    ]
)

print(response.choices[0].message.content)

Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [34]:
def ask_question(question):

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    context = "\n".join(results["documents"][0])

    prompt = f"""
Context:
{context}

Question:
{question}

Answer based only on the context.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role":"user","content":prompt}
        ]
    )

    return response.choices[0].message.content

In [35]:
answer = ask_question(
    "What placement training programs are available?"
)

print(answer)

The context mentions that the college provides exclusive placement training facilities, including:

1. More than four 200+ seating capacity facilities
2. Four 60+ seating capacity training rooms

It can accommodate about 1000 students to undergo dedicated hands-on placement training. However, specific details about the placement training programs are not provided.


In [36]:
def ask_question(question):

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    context = "\n".join(results["documents"][0])

    prompt = f"""
Context:
{context}

Question:
{question}

Answer only using the provided context.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [37]:
print(ask_question("What is SECE?"))

print(ask_question("Tell me about the CSE department"))

print(ask_question("What facilities are available for students?"))

SECE stands for Sri Eshwar Engineering College.
The CSE department has highly experienced and capable faculty who are trained by the Industry. It has state of the art infrastructure and Industry Powered Nextgen Laboratories in AI-ML, Cloud Technologies, Data Science, and Design Thinking. The department uses software technologies including C, Python for Data Science, and Java Full Stack.
The facilities available for students include:

1. Uniquely designed spaces that inspire students to work on their projects.
2. Placement Training Facilities with:
   - Four 200+ seating capacity facilities
   - Four 60+ seating capacity training rooms
   - Capacity to accommodate about 1000 students for dedicated hands-on placement training
3. Corporate style offices and spaces for students to work on their projects.
4. Beautiful campus ambience with grass and trees.
5. Centres of Excellence that promote multi-disciplinary learning.


In [38]:
from google.colab import files

files.download("website_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>